# RAG Part 2 — Experiment 1: the answer prompt (generator)

Given a question and the top-5 excerpts from the retriever, the generator (`LLM_MODEL`) fills
`alpha_audit.schemas.Answer`: `analysis`, `status` (ANSWERED / PARTIAL / NOT_IN_CORPUS / DECLINED_ADVICE), `answer`
(prose with [E#] markers), `citations` (excerpt id + verbatim quote) and `conflicts`. Then the guards run: the
grounding guard drops citations that are not verbatim quotes, the numeric guard flags figures the cited excerpts do
not contain.

**You write the prompt** (`ANSWER_SYSTEM` / `ANSWER_USER` in `src/alpha_audit/prompts/prompt_templates.py`). This
notebook shows what the model sees (section 1), gives a guide for writing the prompt (section 2), and scores each
version on a **dev** sample of the Q&A set: deterministic checks (section 4) and an LLM judge (section 5).

**Workflow per version:** edit `prompt_templates.py` → bump `ANSWER_VERSION` → re-run sections 3–6 (the module
reloads automatically) → read the errors → one line in the iteration log (section 6) → repeat.

> **Experimental.** The Q&A set was written by Claude Haiku 5.5 and will be rebuilt with another model. The judge is
> `JUDGE_LLM_MODEL` (gemini-3.8-flash), a third model family; its scores count only once it agrees with your labels
> (section 5b). Keep the **test** split untouched until the prompt is final.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

os.environ.setdefault("HF_HUB_OFFLINE", "1")  # models are cached; skip the Hub check
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
sys.path.insert(0, "..")

import random
from collections import Counter

import pandas as pd

from alpha_audit.config import get_settings
from alpha_audit.llm.generator import Generator
from alpha_audit.prompts import prompt_templates
from alpha_audit.retrieval import Retriever
from eval import answer_eval as AE
from eval.common import load_goldset

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)
S = get_settings()
S.check_judge_independent()
LOGS = S.path("logs")
print("generator:", S.llm.model, "| reasoning effort:", S.llm.answer_reasoning_effort)
print("judge:    ", S.judge_llm.model, "| eval-set writer:", S.eval_llm.model)
print("retrieval:", {k: S.retrieval.model_dump()[k] for k in ("k_fetch", "bm25_k", "k", "not_in_corpus_threshold")})

/Users/zhaoweikennethlim/Desktop/fin-audit-guard-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


generator: gpt-6-luna | reasoning effort: medium
judge:     gemini-3.8-flash | eval-set writer: azure.claude-haiku-5-5
retrieval: {'k_fetch': 50, 'bm25_k': 50, 'k': 10, 'not_in_corpus_threshold': 0.2}


## 1. What the LLM will see

Four dev questions: a plain fact, a false premise, a "fund covered but figure not stated" question, and a two-article
synthesis. Look at what retrieval returns: **this is all the model gets.** `gate` shows whether the retrieval gate
lets the question reach the LLM (top reranker score ≥ 0.20); expt 4 found that most not-in-corpus questions *do*
reach it, so the prompt must recognise a missing figure itself.

In [2]:
items = load_goldset("../eval/qa_goldset.jsonl")
by_qid = {it["qid"]: it for it in items}
retriever = Retriever()
generator = Generator(retriever, llm=AE.CachedLLM(LOGS / "part2_answers.jsonl", S.llm))

dev = [it for it in items if it["split"] == "dev"]
examples = [next(it for it in dev if it["category"] == c) for c in ("fact", "false_premise", "not_in_corpus", "synthesis")]
ex_ret = AE.retrieve_cached(retriever, examples, LOGS / "part2_retrieval.jsonl")
for it in examples:
    res = ex_ret[it["qid"]]
    print(f"\n[{it['qid']}] {it['question']}")
    print(f"   expect={it['expect']} key_facts={it['key_facts']} premise={it['premise_values']}")
    print(f"   top score {res.top_score:.2f} -> gate {'passes' if retriever.in_corpus(res) else 'BLOCKS (no LLM call)'}")
    for i, e in enumerate(res.evidence, 1):
        print(f"   E{i} score={e.score:.2f} gold={it['gold_chunk_ids'].get(e.chunk_id, 0)} | {e.heading[:100]}")


[fact-005] What is the annual expense ratio charged by the Vanguard Total Stock Market ETF (VTI)?
   expect=answer key_facts=['0.03%'] premise=[]
   top score 1.00 -> gate passes
   E1 score=1.00 gold=2 | VTI ETF Review: Vanguard Total Stock Market Index Fund ETF › Low Expense Ratio
   E2 score=1.00 gold=2 | VTI ETF Review: Vanguard Total Stock Market Index Fund ETF › Key Facts
   E3 score=1.00 gold=2 | VTI ETF vs. VOO ETF: Comparing Vanguard's Top Index Funds › Fee Structure and Expense Ratios of VTI
   E4 score=1.00 gold=2 | VTI ETF vs. VOO ETF: Comparing Vanguard's Top Index Funds › Expense Ratios and Their Impact on Retur
   E5 score=0.98 gold=2 | VTI ETF vs. VOO ETF: Comparing Vanguard's Top Index Funds › Advantages of Investing in VTI ETF
   E6 score=0.97 gold=1 | VTI ETF Review: Vanguard Total Stock Market Index Fund ETF › VTI ETF: 12-Question Expert FAQ for Van
   E7 score=0.96 gold=0 | VTI ETF vs. VOO ETF: Comparing Vanguard's Top Index Funds › Fee Structure and Expense Ratio

In [3]:
# The evidence block exactly as the prompt receives it (format_evidence in prompt_templates.py)
print(prompt_templates.format_evidence(ex_ret[examples[1]["qid"]].evidence)[:3000])

<evidence id="E1" source="VTI ETF vs. VOO ETF: Comparing Vanguard's Top Index Funds › Fee Structure and Expense Ratios">
In line with Vanguard’s philosophy of low-cost investing, VOO boasts an exceptionally low expense ratio of 0.03, especially when compared to actively managed funds and even other ETFs tracking the same index. This cost efficiency is a cornerstone of VOO’s appeal. Lower expense ratios translate directly to better net returns for investors, a critical factor in long-term investment success.
Vanguard's investor-first approach is further exemplified in VOO’s structure. The company's client-owned structure means that fund shareholders are the owners of Vanguard, aligning the company's interests with those of its investors. This unique model ensures that VOO's focus remains on delivering value and performance to its shareholders.
</evidence>

<evidence id="E2" source="VTI ETF vs. VOO ETF: Comparing Vanguard's Top Index Funds › Expense Ratios and Their Impact on Returns">
B

## 2. Writing the answer prompt — a guide

An answer prompt is a **specification**: what to produce, from what, and how to decide the hard cases. The model is
fluent already; what it lacks is *your* rulings. Write for a smart new colleague who has never seen this project.

### Decide first: the four statuses
Write down, in your own words, when each status applies — the model will follow your definitions, so they must not
overlap. Questions your definitions should settle:
- A question with two parts where the excerpts cover only one — ANSWERED or PARTIAL?
- The excerpts discuss the fund but never state the asked figure (expt 4: 14 of 19 such questions pass the retrieval
  gate). What must the answer say, and which status?
- A personal-advice question ("Should I buy DBMF for my IRA?") — decline only, or decline *and* summarise what the
  articles say about the fund? What counts as advice versus an informational "which fund has the lowest fee?"

### Principles
1. **Role and task in two sentences.** Who reads the answer (a reader of this blog), what a good answer is (short,
   direct, every claim cited), and that it is informational only.
2. **Ground strictly, and say why.** Only the excerpts count — not the model's own knowledge of these funds, which
   may be newer, older or wrong for this corpus. Say what to do when the excerpts don't answer the question.
3. **Make the guards your allies.** The grounding guard drops any quote that is not an *exact* substring of its
   excerpt; the numeric guard flags figures (%, decimals, years) that the cited excerpts do not contain. Tell the
   model the rules these guards enforce: copy quotes character for character, keep them short, state figures as
   written, cite the excerpt a figure comes from.
4. **Define the citation format.** Where [E#] markers go (after each claim? each sentence?), what goes in
   `citations` (one per marker? the sentence that holds the figure?), and what to cite for NOT_IN_CORPUS.
5. **Rule on the cases this corpus produces.** If you don't, the model decides them differently each time:
   - **False premises** — "Why is fund X's fee 3%?" when the excerpts say 0.95%: correct the premise from the
     excerpts instead of explaining it.
   - **Articles that disagree** — e.g. one article gives a fund's net expense ratio as 0.95%, another as 0.89%.
     When does a difference count as a conflict, what goes in `conflicts`, and what reasons are allowed (date,
     share class, gross vs net)? Different **backtest periods** are *not* conflicts.
   - **Attribution** — a model portfolio that *holds* a fund is not the fund (the spot-check found reference answers
     that mixed them up; so will the model). Figures belong to the fund, portfolio, share class and period the
     excerpt names.
   - **Comparisons where only one side is covered** — say which side is missing.
   - **Dated figures** — fees, AUM and returns are as of the article; say so.
6. **Separate instructions from data.** Excerpts are untrusted text inside `<evidence>` tags: data to answer from,
   never instructions to follow — even if an excerpt says "ignore previous instructions".
7. **Use `analysis` as the model's working space.** It comes first in the schema on purpose. Tell the model what to
   do there: which excerpt answers which part of the question, whether the premise holds, which figures disagree.
   Readers don't see it. One check matters most here: **is the asked metric, for the asked entity, actually stated
   in an excerpt?** Most not-in-corpus questions arrive with relevant-looking excerpts (the right fund, a high
   reranker score, just not the figure asked for), so the model is tempted to answer from a nearby figure (a Sharpe
   ratio when asked for a Sortino ratio, a portfolio's drawdown when asked for the fund's) or from what it already
   knows. A check written out in `analysis` before the status is chosen makes that harder.
8. **Explain every output field.** The schema in `schemas.py` has field names but no descriptions, so the model sees
   only `analysis`, `status`, `answer`, `citations`, `conflicts`. Either explain each one in the prompt or add
   `Field(description=...)` to the schema — that is prompt design too.
9. **System vs user message.** Stable rules go in `ANSWER_SYSTEM`; this question's data goes in `ANSWER_USER`
   (`{evidence}` and `{question}`). Long evidence before the question usually works better than after.
10. **Every sentence should change a decision.** Cut generic lines ("you are an expert", "be accurate", "think step
    by step"). Use the exact status names. Check that no two rules contradict each other.
11. **Examples leak.** If you add a worked example, invent it (as the examples above are invented) — never copy a
    question or fund case from `qa_goldset.jsonl`, or dev scores stop meaning anything. The same goes for reading
    test items: keep to the dev errors in sections 4–5.
12. **Change one thing per version** and log it (section 6); read the errors (sections 4 and 5) before the next.

In [4]:
# The rendered prompt for one dev question: always read the *final* text the model receives
try:
    system, user = prompt_templates.render_answer(examples[1]["question"], ex_ret[examples[1]["qid"]].evidence)
    print(f"=== SYSTEM ({prompt_templates.ANSWER_VERSION}) ===\n{system}\n\n=== USER ===\n{user[:3000]}")
except NotImplementedError as e:
    print("Prompt not written yet:", e)

=== SYSTEM (v2) ===
You are a financial research assistant that answers questions about
articles from the investment blog Picture Perfect Portfolios, using
only excerpts provided with each question. Your task is to give readers
a direct answer to the question first, followed only by the details it
needs, with facts and figures stated in the excerpts and citations
from them after every claim. 

Set status to one of:
- ANSWERED: the excerpts state everything the question asks for.
- PARTIAL: the excerpts state some of what the question asks for, but not all of it.
- NOT_IN_CORPUS: the excerpts state none of what the question asks for, even if they mention the fund.
- DECLINED_ADVICE: the question asks for a personal recommendation. In the answer, decline the recommendation, then say what the excerpts report about the fund, with citations.

If a question asks for a personal recommendation, use DECLINED_ADVICE even if the excerpts could fit another status.

Use only context from the excerp

## 3. Dev run

**Choose how many calls to spend.** In the next cell, `MAX_NEW_CALLS` limits the generator calls each time you run the
cell after it, and `MAX_JUDGE_CALLS` does the same for section 5 (0 skips the judge; `None` means no limit). Answers
are cached by exact prompt text + model (`logs/part2_answers.jsonl`), so **re-running continues where you stopped**:
finished questions are free, and up to `MAX_NEW_CALLS` new ones are added. The questions take turns by category, so
even a small limit covers each type. Sections 4–5 score whatever has been answered so far; section 6 logs a version
only once all its questions are answered.

`SAMPLE = "quick"` takes a fixed stratified sample (~50 questions, all synthesis/conflict/advice dev items) for fast
iteration; `"full"` runs all 140 dev questions. Retrieval runs once and is cached (`logs/part2_retrieval.jsonl`). A
quick run is ~47 LLM calls in total (gated questions cost none): a few minutes at the proxy's limit of 20 per minute.

In [10]:
SAMPLE, SEED = "quick", 0
MAX_NEW_CALLS = None  # generator calls per run of the next cell; None = no limit
MAX_JUDGE_CALLS = None  # judge calls per run of section 5; 0 = skip the judge, None = no limit

QUICK = {"fact": 8, "false_premise": 8, "faq": 10, "synthesis": 99, "conflict": 99, "not_in_corpus": 8, "advice": 99}
rng = random.Random(SEED)
groups = []
for cat, n in QUICK.items():
    pool = sorted((it for it in dev if it["category"] == cat), key=lambda it: it["qid"])
    groups.append(pool if SAMPLE == "full" else rng.sample(pool, min(n, len(pool))))
sample = [g[i] for i in range(max(map(len, groups))) for g in groups if i < len(g)]  # categories take turns
retrievals = AE.retrieve_cached(retriever, sample, LOGS / "part2_retrieval.jsonl")
gated = sum(not retriever.in_corpus(retrievals[it["qid"]]) for it in sample)
print(f"{len(sample)} dev questions ({SAMPLE}):", dict(Counter(it["category"] for it in sample)),
      f"| {gated} stopped by the gate (no LLM call)")

51 dev questions (quick): {'fact': 8, 'false_premise': 8, 'faq': 10, 'synthesis': 8, 'conflict': 1, 'not_in_corpus': 8, 'advice': 8} | 4 stopped by the gate (no LLM call)


In [11]:
judged = pd.DataFrame()  # section 5 fills this; empty until then, so verdicts from an earlier prompt are never reused
try:
    generator.llm.remaining = MAX_NEW_CALLS
    calls_before = generator.llm.usage["calls"]
    runs = AE.run_answers(generator, sample, retrievals)
    scores = AE.score_all(sample, runs)
    answered = int(scores.status.notna().sum())
    print(f"prompt {prompt_templates.ANSWER_VERSION}: {answered} of {len(sample)} questions answered "
          f"| {generator.llm.usage['calls'] - calls_before} new generator calls this run")
    if answered < len(sample):
        print(f"{len(sample) - answered} still to do: re-run this cell to continue")
    if not answered:
        runs = scores = None
except NotImplementedError as e:
    runs = scores = None
    print("Prompt not written yet:", e)

prompt v2: 51 of 51 questions answered | 0 new generator calls this run


## 4. Deterministic scores

- `status_ok` — the status the item expects (fact / false premise / FAQ / synthesis / conflict → ANSWERED,
  not_in_corpus → NOT_IN_CORPUS, advice → DECLINED_ADVICE; advice refused by the gate also counts, see
  `advice_gated`).
- `key_fact_recall` — share of the item's key facts in the answer (figures by value). FAQ items are excluded: their
  key facts are hints and the judge scores them.
- `conflict_found` — a Conflict reported with both values. `cites_gold` — a cited excerpt states the fact.
- `unexpected_conflict` — a Conflict reported on a question not labelled as a conflict. Some may be real, so read
  them in 4b: this is the only score that surfaces false conflicts.
- `dropped` / `numeric_flags` — what the guards caught. Many dropped quotes = the prompt's quoting rule is unclear.
- **Not-in-corpus check.** Look at the `not_in_corpus` row of `status_ok` (most of these questions get past the
  gate, so this row measures your prompt, not retrieval), and at any `numeric_flags` on those questions: a flagged
  figure on a not-in-corpus question is almost certainly made up.

In [12]:
if scores is not None:
    ok = scores[scores.status.notna()]
    cols = ["status_ok", "key_fact_recall", "conflict_found", "unexpected_conflict", "cites_gold", "dropped", "numeric_flags", "seconds"]
    table = ok.groupby("category")[cols].agg(lambda s: s.dropna().astype(float).mean() if s.notna().any() else None)
    table.insert(0, "n", ok.groupby("category").size())
    display(table.round(2))
    print("expected status (rows) vs answered status (columns):")
    display(pd.crosstab(ok.expect, ok.status, margins=True))
    flags = Counter(k for ks in ok.flag_kinds for k in ks)
    print("guard flags:", dict(flags) or "none")

,n,status_ok,key_fact_recall,conflict_found,unexpected_conflict,cites_gold,dropped,numeric_flags,seconds
category,,,,,,,,,
advice,8,1.00,NaN,NaN,0.12,NaN,0.0,0.00,3.08
conflict,1,1.00,0.50,0.0,NaN,1.00,0.0,0.00,2.56
fact,8,0.88,1.00,NaN,0.00,0.88,0.0,0.12,3.79
false_premise,8,0.88,1.00,NaN,0.12,1.00,0.0,0.00,6.31
faq,10,0.90,NaN,NaN,0.00,1.00,0.0,0.00,3.88
not_in_corpus,8,0.88,NaN,NaN,0.00,NaN,0.0,0.00,3.71
synthesis,8,0.38,0.46,NaN,0.00,0.38,0.0,0.12,5.26


expected status (rows) vs answered status (columns):


status,ANSWERED,DECLINED_ADVICE,NOT_IN_CORPUS,PARTIAL,All
expect,,,,,
answer,19,0,0,7,26
correct_premise,7,0,0,1,8
decline_advice,0,4,4,0,8
not_in_corpus,0,0,7,1,8
report_conflict,1,0,0,0,1
All,27,4,11,9,51


guard flags: {'unsupported_percent': 2}


### 4b. Errors to read

Every item with the wrong status, missing key facts, a guard flag or an unexpected conflict — with the answer, the flags and the excerpts it
had. **This is where the next prompt change comes from.**

In [8]:
if scores is not None:
    res_by = {it["qid"]: res for it, (res, _) in zip(sample, runs)}
    bad = scores[scores.status_ok.eq(False) | (scores.key_fact_recall < 1)
                 | (scores.grounding_flags > 0) | (scores.numeric_flags > 0) | scores.unexpected_conflict.eq(True)]
    print(f"{len(bad)} of {answered} answers to review\n")
    for _, r in bad.iterrows():
        it, res = by_qid[r.qid], res_by[r.qid]
        print(f"[{r.qid}] expect={r.expect} status={r.status} gated={r.gated} top={r.top_score}")
        print("   Q:", it["question"])
        if r.missing_facts:
            print("   missing key facts:", r.missing_facts)
        if it["premise_values"]:
            print("   wrong premise:", it["premise_values"])
        print("   A:", (r.answer or "")[:700])
        for f in res.flags:
            print(f"   flag {f.guard}/{f.kind}: {f.detail[:160]}")
        if r.unexpected_conflict is True:
            for conflict in res.answer.conflicts:
                values = " | ".join(f"{p.evidence_id}: {p.quote[:120]}" for p in conflict.positions)
                print(f"   unexpected conflict: {conflict.topic} ({conflict.likely_reason}) {values}")
        if not res.gated:
            print("   analysis:", res.raw_answer.analysis[:400])
        for i, e in enumerate(res.retrieval.evidence[:3], 1):
            print(f"   E{i} gold={it['gold_chunk_ids'].get(e.chunk_id, 0)} {e.heading[:90]}")
        print()

17 of 51 answers to review

[synthesis-019] expect=answer status=ANSWERED gated=False top=0.934
   Q: How much stock and bond exposure does a single dollar invested in PSLDX provide, and how do the two discussions of that fund compare?
   missing key facts: ['$1 of S&P 500', '100% stocks', '100% bonds']
   A: A dollar invested in PSLDX provides roughly $1 of stock exposure and $1 of bond exposure—about $2 total, or 200% notional exposure. [E2] [E4]

The Jack Burns discussion emphasizes the leverage and fair pricing of PSLDX’s 200% notional exposure, saying the investor is mostly invested for those features. [E1] The fund review describes PSLDX as a capital-efficient “100/100” stocks-and-bonds design, but cautions that it may be better used as a diversified portfolio building block than as a total-portfolio solution. [E4] [E5] [E7]
   analysis: The excerpts state the exposure for PSLDX: approximately $1 of stock exposure and $1 of bond exposure per dollar invested, or 200% notional expo

## 5. LLM judge

- **correctness** — items with a reference answer (FAQ: the article's own FAQ answer; synthesis: the approved
  reference): correct / partially_correct / incorrect.
- **faithfulness** — every answer that says something: is each statement supported by the excerpts shown?
  faithful / minor_issues / unfaithful.

Verdicts are cached in `logs/part2_judge.jsonl`. The judge prompts are in `eval/judge.py`.

In [13]:
judge_llm = AE.CachedLLM(LOGS / "part2_judge.jsonl", S.judge_llm)
judge_llm.remaining = MAX_JUDGE_CALLS
if runs is not None:
    judged = AE.judge_all(sample, runs, judge_llm)
    print("judge LLM:", judge_llm.usage)
    if len(judged):
        display(pd.crosstab([judged.task, judged.category], judged.verdict, margins=True))
    for _, r in judged[judged.verdict.isin(["incorrect", "partially_correct", "unfaithful"])].iterrows():
        print(f"\n[{r.qid}] {r.task}: {r.verdict}\n   Q: {by_qid[r.qid]['question']}")
        for d in r.details[:4]:
            print("   -", d[:200])
        print("   judge:", r.reasoning[:400])

judge LLM: {'calls': 59, 'cached': 0, 'input_tokens': 133904, 'output_tokens': 24278}


verdict                     correct  faithful  incorrect  partially_correct  All
task         category                                                           
correctness  conflict             0         0          0                  1    1
             faq                 10         0          0                  0   10
             synthesis            2         0          4                  2    8
faithfulness advice               0         4          0                  0    4
             conflict             0         1          0                  0    1
             fact                 0         8          0                  0    8
             false_premise        0         8          0                  0    8
             faq                  0        10          0                  0   10
             not_in_corpus        0         1          0                  0    1
             synthesis            0         8          0                  0    8
All                              12        40          4                  3   59


[conflict-001] correctness: partially_correct
   Q: What is the net expense ratio of the QVAL ETF?
   - Another source in the text places QVAL's net expense ratio in the 0.49 to 0.59 range.
   judge: The reference notes that the source documents provide conflicting information: one source states the net expense ratio is 0.39, while another places QVAL within a 0.49 to 0.59 range. The answer only reports 0.39% and fails to mention the alternative range.

[synthesis-016] correctness: partially_correct
   Q: How do the exposures of RSBT and HCMT compare, and what combined share of the Tactical Tornado Portfolio do they make up?
   - RSBT holds 100% Aggregate Bonds and 100% Managed Futures exposure.
   judge: The answer correctly identifies the combined share in the Tactical Tornado Portfolio (60%, from 40% RSBT and 20% HCMT) and accurately details HCMT's exposure (200% US equities or cash). However, it misses the specific exposure percentages for RSBT (100% Aggregate Bonds plus 100% Mana

### 5b. Validate the judge before trusting it

`logs/part2_judge_spotcheck.csv` collects judge verdicts across tasks and verdicts (25 by default). Fill
`human_ok` with **y** (you agree with the verdict) or **n** (and a `human_note`), save, and re-run this cell. Rows you
labelled keep their labels; new rows are added until the file holds `n`. Trust the judge's scores only once agreement
is high (≳ 85%) on both tasks.

In [17]:
if runs is not None and len(judged):
    spot = AE.judge_spotcheck(judged, sample, runs, LOGS / "part2_judge_spotcheck_v2_labelled.csv", n=25)
    print("judge spot-check file:", spot)
    agreement = AE.judge_agreement(spot)
    display(agreement if len(agreement) else "no labels yet")

judge spot-check file: /Users/zhaoweikennethlim/Desktop/fin-audit-guard-rag/logs/part2_judge_spotcheck_v2_labelled.csv


,agreement,labelled
task,,
correctness,1.0,16
faithfulness,1.0,9


## 6. Iteration log

One row per prompt version (with the prompt's hash, so an edit without a version bump shows up), written to
`logs/part2_iterations.csv`. Add a note on what changed; then record the takeaway in the table below.

In [16]:
if runs is not None and answered == len(sample):
    summary = AE.summarise(scores, judged)
    log = AE.log_iteration(LOGS / "part2_iterations.csv", summary | {"sample": SAMPLE}, note="v2 disagreement rule + k=10")
    display(log.tail(10))
elif runs is not None:
    print(f"Not logged yet: {answered} of {len(sample)} questions answered.")

,logged,prompt_version,prompt_hash,model,reasoning_effort,judge,n,status_ok,key_fact_recall,conflict_found,...,correct+partial (judge),faithful (judge),faithful+minor (judge),answers with dropped citations,answers with numeric flags,unexpected conflicts,llm seconds (mean),tokens in/out (mean),sample,note
0,2026-11-07T16:29,v1,526c9314fc,gpt-6-luna,medium,gemini-3.8-flash j1,51,0.843,0.857,0.0,...,0.737,1.0,1.0,0.039,0.020,3,10.1,1975/471,quick,v1 baseline
1,2026-11-07T20:40,v2,feee8e334b,gpt-6-luna,medium,gemini-3.8-flash j1,51,0.824,0.807,0.0,...,0.789,1.0,1.0,0.000,0.039,2,4.7,3202/516,quick,v2 disagreement rule + k=10


In [32]:
if runs is not None and answered == len(sample):
    summary = AE.summarise(scores, judged)
    log = AE.log_iteration(LOGS / "part2_iterations.csv", summary | {"sample": SAMPLE}, note="v2 disagreement rule + k=10")
    display(log.tail(10))
elif runs is not None:
    print(f"Not logged yet: {answered} of {len(sample)} questions answered.")

,logged,prompt_version,prompt_hash,model,reasoning_effort,judge,n,status_ok,key_fact_recall,conflict_found,...,correct+partial (judge),faithful (judge),faithful+minor (judge),answers with dropped citations,answers with numeric flags,unexpected conflicts,llm seconds (mean),tokens in/out (mean),sample,note
0,2026-11-07T16:29,v1,526c9314fc,gpt-6-luna,medium,gemini-3.8-flash j1,51,0.843,0.857,0.0,...,0.737,1.0,1.0,0.039,0.02,3,10.1,1975/471,quick,v1 baseline


| version | change | status_ok | key-fact recall | correct (judge) | faithful (judge) | notes (errors fixed / introduced) |
|---|---|---|---|---|---|---|
| v1 | |0.843 |0.857 |None |None | |